In [ ]:
from utils_monoBP_single import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.pyplot as plt
import math
import pandas as pd
import time
import sys
from scipy.stats import ks_2samp
import ot
from torch.distributions import MultivariateNormal

# Averaged Results

In [ ]:
task_set = [i for i in range(10)]

In [ ]:
len(task_set)

In [ ]:
KS_single = torch.zeros(len(task_set), 101)
Cvg_single = torch.zeros(len(task_set), 101)
Width_single = torch.zeros(len(task_set), 101)
W1_single = torch.zeros(len(task_set), 101)

KS_single_nondiag = torch.zeros(len(task_set), 101)
Cvg_single_nondiag = torch.zeros(len(task_set), 101)
Width_single_nondiag = torch.zeros(len(task_set), 101)
W1_single_nondiag = torch.zeros(len(task_set), 101)

Width_gibbs = torch.zeros(len(task_set), 101)
Cvg_gibbs = torch.zeros(len(task_set), 101)

In [ ]:
for j in tqdm(range(len(task_set))):
    task_id = task_set[j]

    pred_ys_gibbs = torch.tensor( pd.read_csv(f'sample_res/pred_ys_gibbs_task{task_id}.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()
    
    pred_ys_nmodel= torch.tensor( pd.read_csv(f'sample_res_DebReg_fisher_cross/pred_ys_ann_task{task_id}.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()
    
    pred_ys_nmodel_nondiag= torch.tensor( pd.read_csv(f'sample_res_DebReg_fisher_cross_nondiag/pred_ys_ann_task{task_id}.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()

    
    A = get_A(M)
    grids = torch.arange(0, 1.01, 0.01)
    psi_grids = get_psi(grids, M)


    ######## Calculation begins
    
    # KS error
    for i in range(pred_ys_gibbs.shape[0]):
        KS_single[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_nmodel[i]).statistic
        KS_single_nondiag[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_nmodel_nondiag[i]).statistic

    # W1 error
    for i in range(pred_ys_gibbs.shape[0]):
        W1_single[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_nmodel[i])
        W1_single_nondiag[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_nmodel_nondiag[i])
    
    # CI width
    for i in range(pred_ys_gibbs.shape[0]):
        Width_single[j, i] = torch.quantile(pred_ys_nmodel[i], 0.975) - torch.quantile(pred_ys_nmodel[i], 0.025)

        Width_single_nondiag[j, i] = torch.quantile(pred_ys_nmodel_nondiag[i], 0.975) - torch.quantile(pred_ys_nmodel_nondiag[i], 0.025)

        Width_gibbs[j, i] = torch.quantile(pred_ys_gibbs[i], 0.975) - torch.quantile(pred_ys_gibbs[i], 0.025)
        
    # Coverage
    true_mean = torch.tanh(4.0 * grids - 2.0)
    for i in range(pred_ys_gibbs.shape[0]):
        Cvg_single[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_nmodel[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_nmodel[i], 0.975) )
        Cvg_single_nondiag[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_nmodel_nondiag[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_nmodel_nondiag[i], 0.975) )
        Cvg_gibbs[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_gibbs[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_gibbs[i], 0.975) )

In [ ]:
KS_single.mean().item(), W1_single.mean().item(), Cvg_single.mean().item(), Width_single.mean().item()

In [ ]:
KS_single_nondiag.mean().item(), W1_single_nondiag.mean().item(), Cvg_single_nondiag.mean().item(), Width_single_nondiag.mean().item()